# Machine Learning — Lab 6
## Logistic Regression: Forward Pass, Loss, and Learning

**Main Course Learning Outcomes — CLO3 and CLO4**

- **CLO3:** Construct and analyze machine learning models by applying algorithmic principles and internal computations.
- **CLO4:** Apply supervised learning techniques to solve practical problems and interpret their outcomes.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** A correct classifier is not enough. You must show that you understand how logistic regression moves from **features → score → probability → loss → gradient → parameter update**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Binary-classification formulation | 10 min | Identify inputs, target, positive class, and prediction goal |
| 2. Sigmoid and forward pass | 20 min | Compute scores, probabilities, and class predictions |
| 3. Binary cross-entropy | 20 min | Implement and interpret log loss |
| 4. Manual gradient update | 30 min | Compute $p-y$, gradients, and one full parameter update |
| 5. Train logistic regression | 20 min | Fit a model and interpret coefficients |
| 6. Decision boundary & probability reasoning | 10 min | Connect $w^\top x+b=0$ to classification |
| 7. Challenge, debugging & viva | 10 min | Diagnose mistakes and explain model behavior |
| **Total** | **120 min** | |

### Main idea

$$
\boxed{
x
\rightarrow
z=w^\top x+b
\rightarrow
p=\sigma(z)
\rightarrow
L
\rightarrow
\nabla L
\rightarrow
\text{update}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. formulate a binary-classification task;
2. implement the sigmoid function;
3. compute the logistic-regression score $z=w^\top x+b$;
4. convert $z$ into a probability;
5. use a threshold to obtain a class prediction;
6. compute binary cross-entropy loss;
7. explain why confident wrong predictions receive high loss;
8. compute one gradient-descent update manually;
9. fit logistic regression using `scikit-learn`;
10. interpret coefficient signs and a simple decision boundary.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Machine Learning Lab 6 environment ready.")

# Part I — Binary Classification Problem

We will model **customer subscription cancellation**.

Target:

```text
cancelled
```

where:

- `1` = customer cancelled;
- `0` = customer remained subscribed.

Candidate features include:

- `monthly_usage_hours`
- `support_tickets`
- `months_with_service`
- `monthly_fee`
- `late_payments`

This is a supervised **binary classification** problem.

In [ ]:
rng = np.random.default_rng(3452)
n = 420

monthly_usage_hours = np.clip(rng.normal(28, 11, n), 2, 70)
support_tickets = np.clip(rng.poisson(2.2, n), 0, 9)
months_with_service = np.clip(rng.gamma(3.0, 9.0, n), 1, 72)
monthly_fee = np.clip(rng.normal(115, 28, n), 45, 220)
late_payments = np.clip(rng.poisson(1.3, n), 0, 7)

# Latent cancellation probability.
logit = (
    -0.075 * monthly_usage_hours
    + 0.50 * support_tickets
    - 0.035 * months_with_service
    + 0.018 * monthly_fee
    + 0.55 * late_payments
    - 1.20
)

prob = 1 / (1 + np.exp(-logit))
cancelled = rng.binomial(1, prob, size=n)

df_master = pd.DataFrame({
    "monthly_usage_hours": np.round(monthly_usage_hours, 1),
    "support_tickets": support_tickets,
    "months_with_service": np.round(months_with_service, 1),
    "monthly_fee": np.round(monthly_fee, 1),
    "late_payments": late_payments,
    "cancelled": cancelled,
})

print("Master dataset shape:", df_master.shape)
display(df_master.head())
print("\nTarget distribution:")
display(df_master["cancelled"].value_counts().rename(index={0:"not_cancelled", 1:"cancelled"}))

## Task 1.1 — Personalized Dataset

Enter the **last four digits** of your student ID.

Your ID determines a reproducible working sample of 320 customers.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 6000 + (STUDENT_ID_LAST4 % 4000)

df = df_master.sample(
    n=320,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your logistic-regression seed:", SEED)
print("Working dataset shape:", df.shape)

## Task 1.2 — Formulate the Problem

Complete:

- **Learning paradigm:**  
- **Task type:**  
- **Positive class:**  
- **Negative class:**  
- **Target $y$:**  
- **Input features $X$:**  
- **Model output before thresholding:**  
- **Model output after thresholding:**  

Then answer:

> Why is this not a regression task even though the algorithm is called **logistic regression**?

# Part II — Sigmoid and Forward Pass

Logistic regression first computes:

$$
z=w^\top x+b.
$$

Then it maps the score to a probability:

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}.
$$

The output satisfies:

$$
0<p<1.
$$

## Task 2.1 — Implement Sigmoid

Complete the function.

In [ ]:
def sigmoid(z):
    # TODO: implement the sigmoid function.
    result = None
    return result

In [ ]:
# Self-check after implementation.
test_z = np.array([-2.0, 0.0, 2.0])
test_p = sigmoid(test_z)

assert np.allclose(
    test_p,
    np.array([0.11920292, 0.5, 0.88079708]),
    atol=1e-7
)

print("Sigmoid function test passed.")

## Task 2.2 — Predict the Sigmoid Behavior

Before running the next cell, complete:

| $z$ | Expected probability behavior |
|---:|---|
| very negative |  |
| $-1$ |  |
| $0$ |  |
| $+1$ |  |
| very positive |  |

Then state:

1. Why does $z=0$ correspond to probability $0.5$?
2. Why does sigmoid never produce values below 0 or above 1?

In [ ]:
z_grid = np.linspace(-8, 8, 400)

plt.figure(figsize=(7, 4))
plt.plot(z_grid, sigmoid(z_grid))
plt.axhline(0.5, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("Linear score z")
plt.ylabel("Sigmoid probability")
plt.title("Sigmoid Function")
plt.show()

## Task 2.3 — Implement a Forward Pass

For one example:

$$
z=w^\top x+b
$$

$$
p=\sigma(z).
$$

Complete the function.

In [ ]:
def logistic_forward(x, w, b):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    # TODO: compute z and p.
    z = None
    p = None

    return z, p

In [ ]:
# Self-check.
x_check = np.array([2.0, 1.0])
w_check = np.array([0.8, -0.4])
b_check = 0.2

z_check, p_check = logistic_forward(x_check, w_check, b_check)

assert abs(z_check - 1.4) < 1e-12
assert abs(p_check - 0.8021838886) < 1e-7

print("Forward-pass test passed.")

## Task 2.4 — Manual Forward Pass

Given:

$$
x=
\begin{bmatrix}
2\\
1
\end{bmatrix},
\qquad
w=
\begin{bmatrix}
0.8\\
-0.4
\end{bmatrix},
\qquad
b=0.2,
$$

calculate manually:

1. $z$;
2. $p=\sigma(z)$;
3. predicted class at threshold $0.5$;
4. predicted class at threshold $0.85$.

**Your calculation:**

In [ ]:
z_demo, p_demo = logistic_forward(
    np.array([2.0, 1.0]),
    np.array([0.8, -0.4]),
    0.2
)

print(f"z = {z_demo:.4f}")
print(f"p = {p_demo:.4f}")
print("Class at threshold 0.5:", int(p_demo >= 0.5))
print("Class at threshold 0.85:", int(p_demo >= 0.85))

# Part III — Binary Cross-Entropy Loss

For one example:

$$
L(y,p)
=
-\left[
y\log(p)+(1-y)\log(1-p)
\right].
$$

If $y=1$:

$$
L=-\log(p).
$$

If $y=0$:

$$
L=-\log(1-p).
$$

## Task 3.1 — Predict the Loss

Without calculating exact values, rank the following from **lowest loss** to **highest loss**.

| True label $y$ | Predicted $p$ |
|---:|---:|
| 1 | 0.95 |
| 1 | 0.55 |
| 1 | 0.10 |
| 0 | 0.05 |
| 0 | 0.80 |

Explain why a **confident wrong prediction** should receive a high loss.

## Task 3.2 — Implement Binary Cross-Entropy

Clip probabilities slightly away from 0 and 1 to avoid computing $\log(0)$.

In [ ]:
def binary_cross_entropy(y, p):
    eps = 1e-12
    p = np.clip(float(p), eps, 1 - eps)
    y = float(y)

    # TODO: implement binary cross-entropy.
    loss = None

    return loss

In [ ]:
assert abs(binary_cross_entropy(1, 0.9) - (-np.log(0.9))) < 1e-12
assert abs(binary_cross_entropy(0, 0.1) - (-np.log(0.9))) < 1e-12

print("Binary cross-entropy function tests passed.")

In [ ]:
examples = pd.DataFrame({
    "y": [1, 1, 1, 0, 0],
    "p": [0.95, 0.55, 0.10, 0.05, 0.80],
})

examples["loss"] = [
    binary_cross_entropy(y_i, p_i)
    for y_i, p_i in zip(examples["y"], examples["p"])
]

display(examples.round(4))

## Task 3.3 — Interpret the Loss Table

Answer:

1. Which prediction has the smallest loss?
2. Which has the largest loss?
3. Why are $(y=1,p=0.95)$ and $(y=0,p=0.05)$ both good predictions?
4. Why is probability quality richer information than the final class label alone?

# Part IV — Gradients and One Training Update

For logistic regression with sigmoid and binary cross-entropy:

$$
\frac{\partial L}{\partial z}=p-y.
$$

Therefore:

$$
\frac{\partial L}{\partial w_j}
=
(p-y)x_j
$$

and:

$$
\frac{\partial L}{\partial b}
=
p-y.
$$

Gradient descent then applies:

$$
w_j
\leftarrow
w_j-\alpha(p-y)x_j
$$

$$
b
\leftarrow
b-\alpha(p-y).
$$

## Task 4.1 — Interpret $p-y$

Complete:

| Situation | Sign of $p-y$ | Desired change |
|---|---:|---|
| $y=1$, but $p$ is too small |  | increase score |
| $y=0$, but $p$ is too large |  | decrease score |

Explain why $p-y$ acts as an **error signal**.

## Task 4.2 — Implement One-Example Gradients

Complete the function.

In [ ]:
def logistic_gradients_one(x, y, w, b):
    z, p = logistic_forward(x, w, b)

    x = np.asarray(x, dtype=float)

    # TODO: compute error signal, weight gradients, and bias gradient.
    error_signal = None
    dw = None
    db = None

    return z, p, error_signal, dw, db

In [ ]:
x_toy = np.array([2.0, 1.0])
y_toy = 1
w_toy = np.array([0.5, -0.25])
b_toy = 0.1

z_toy, p_toy, err_toy, dw_toy, db_toy = logistic_gradients_one(
    x_toy, y_toy, w_toy, b_toy
)

assert abs(z_toy - 0.85) < 1e-12
assert abs(err_toy - (p_toy - 1)) < 1e-12
assert np.allclose(dw_toy, err_toy * x_toy)
assert abs(db_toy - err_toy) < 1e-12

print("One-example gradient test passed.")

## Task 4.3 — Manual Training Step

Use:

$$
x=
\begin{bmatrix}
2\\
1
\end{bmatrix},
\quad
y=1,
\quad
w=
\begin{bmatrix}
0.5\\
-0.25
\end{bmatrix},
\quad
b=0.1,
\quad
\alpha=0.1.
$$

Calculate manually:

1. score $z$;
2. probability $p$;
3. loss;
4. error signal $p-y$;
5. $\frac{\partial L}{\partial w_1}$;
6. $\frac{\partial L}{\partial w_2}$;
7. $\frac{\partial L}{\partial b}$;
8. updated $w_1$, $w_2$, and $b$.

Do this before running the verification cell.

In [ ]:
learning_rate = 0.1

z_before, p_before, error_signal, dw, db = logistic_gradients_one(
    x_toy, y_toy, w_toy, b_toy
)

loss_before = binary_cross_entropy(y_toy, p_before)

w_new = w_toy - learning_rate * dw
b_new = b_toy - learning_rate * db

z_after, p_after = logistic_forward(
    x_toy, w_new, b_new
)
loss_after = binary_cross_entropy(y_toy, p_after)

print("Before update")
print("z:", round(z_before, 4))
print("p:", round(p_before, 4))
print("loss:", round(loss_before, 4))
print("error signal p-y:", round(error_signal, 4))
print("dw:", np.round(dw, 4))
print("db:", round(db, 4))

print("\nAfter update")
print("w:", np.round(w_new, 4))
print("b:", round(b_new, 4))
print("new p:", round(p_after, 4))
print("new loss:", round(loss_after, 4))

## Task 4.4 — Explain the Update

Answer:

1. Did the probability of the correct class increase?
2. Did the loss decrease?
3. Why did the model need to increase the score for this example?
4. If $y=0$ instead, in what direction should the score move?

## Task 4.5 — Implement One Correct Update

Complete the reusable function.

In [ ]:
def one_logistic_update(x, y, w, b, learning_rate):
    z, p, error_signal, dw, db = logistic_gradients_one(
        x, y, w, b
    )

    # TODO: apply gradient descent.
    w_updated = None
    b_updated = None

    return w_updated, b_updated

In [ ]:
w_test, b_test = one_logistic_update(
    x_toy, y_toy, w_toy, b_toy, 0.1
)

assert np.allclose(w_test, w_new)
assert abs(b_test - b_new) < 1e-12

print("Gradient-descent update test passed.")

# Part V — Train a Real Logistic-Regression Model

We now use the full customer dataset.

The model will use five features and learn:

$$
p(y=1\mid x)
=
\sigma(w^\top x+b).
$$

We will standardize features because:

- features have different units;
- scaling helps optimization;
- coefficient comparison becomes more meaningful.

In [ ]:
feature_cols = [
    "monthly_usage_hours",
    "support_tickets",
    "months_with_service",
    "monthly_fee",
    "late_payments",
]

X = df[feature_cols].copy()
y = df["cancelled"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## Task 5.1 — Predict Coefficient Signs

Before fitting, predict the expected sign of each coefficient.

| Feature | Expected sign | Reason |
|---|---|---|
| `monthly_usage_hours` |  |  |
| `support_tickets` |  |  |
| `months_with_service` |  |  |
| `monthly_fee` |  |  |
| `late_payments` |  |  |

Your reasoning should describe how increasing the feature is expected to affect the probability of cancellation.

In [ ]:
logistic_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=2000,
        random_state=SEED
    )),
])

logistic_model.fit(X_train, y_train)

valid_prob = logistic_model.predict_proba(X_valid)[:, 1]
valid_pred = (valid_prob >= 0.5).astype(int)

valid_accuracy = accuracy_score(y_valid, valid_pred)
valid_logloss = log_loss(y_valid, valid_prob)

print(f"Validation accuracy: {valid_accuracy:.3f}")
print(f"Validation log loss: {valid_logloss:.3f}")

## Task 5.2 — Interpret the Two Evaluation Values

This lab focuses on the learning mechanics, so we use only a light evaluation preview.

Answer:

1. What does validation accuracy measure?
2. What does log loss measure that accuracy does not?
3. Can two models have the same accuracy but different log loss?
4. Why will classification evaluation be studied more deeply in the next lab?

In [ ]:
coef = logistic_model.named_steps["logistic"].coef_[0]
intercept = logistic_model.named_steps["logistic"].intercept_[0]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient_on_standardized_feature": coef
})

print("Intercept:", round(intercept, 4))
display(coef_table.round(4))

## Task 5.3 — Interpret Coefficient Signs

For each feature:

1. compare learned sign with your prediction;
2. explain whether increasing the standardized feature raises or lowers the log-odds of cancellation;
3. identify the strongest positive coefficient;
4. identify the strongest negative coefficient.

### Caution

A coefficient is not automatically causal evidence.

A positive coefficient means the **model** associates higher feature values with higher cancellation log-odds, holding other modeled features fixed.

# Part VI — Manual Prediction from the Fitted Model

We will select one validation customer and reproduce the probability manually.

Because the model was trained on standardized features, we must first apply the scaler.

In [ ]:
row_index = X_valid.index[SEED % len(X_valid)]
x_row = X_valid.loc[[row_index]]

print("Selected validation row:")
display(x_row)
print("True class:", int(y_valid.loc[row_index]))

In [ ]:
scaler = logistic_model.named_steps["scale"]
classifier = logistic_model.named_steps["logistic"]

x_scaled = scaler.transform(x_row)[0]
w_fit = classifier.coef_[0]
b_fit = classifier.intercept_[0]

manual_z = float(np.dot(w_fit, x_scaled) + b_fit)
manual_p = float(sigmoid(manual_z))

library_p = float(logistic_model.predict_proba(x_row)[0, 1])

print("Scaled feature vector:", np.round(x_scaled, 4))
print("Manual z:", round(manual_z, 6))
print("Manual probability:", round(manual_p, 6))
print("Library probability:", round(library_p, 6))

## Task 6.1 — Explain the Manual Prediction

Answer:

1. Why did we scale the row before applying the learned coefficients?
2. Why would using the original unscaled feature values give the wrong result?
3. How close are the manual and library probabilities?
4. What class is predicted at threshold $0.5$?
5. Does probability $0.7$ mean the model is “70% certain” in an absolute sense? Explain carefully.

# Part VII — Decision Boundary

For logistic regression:

$$
z=w^\top x+b.
$$

At threshold $0.5$:

$$
p=0.5
\iff
z=0.
$$

Therefore, the decision boundary is:

$$
w^\top x+b=0.
$$

For two features, this is a line.

## Task 7.1 — Two-Feature Model

We will fit a simple model using:

- `monthly_usage_hours`
- `late_payments`

to visualize a linear decision boundary.

In [ ]:
two_features = [
    "monthly_usage_hours",
    "late_payments"
]

two_model = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=2000,
        random_state=SEED
    )),
])

two_model.fit(
    X_train[two_features],
    y_train
)

two_scaler = two_model.named_steps["scale"]
two_clf = two_model.named_steps["logistic"]

print("Scaled-space coefficients:", two_clf.coef_[0])
print("Scaled-space intercept:", two_clf.intercept_[0])

In [ ]:
# Visualize probability regions in ORIGINAL feature units.
x1_min = X_train["monthly_usage_hours"].min()
x1_max = X_train["monthly_usage_hours"].max()
x2_min = X_train["late_payments"].min()
x2_max = X_train["late_payments"].max()

x1_grid = np.linspace(x1_min, x1_max, 200)
x2_grid = np.linspace(x2_min, x2_max, 200)

xx1, xx2 = np.meshgrid(x1_grid, x2_grid)

grid_df = pd.DataFrame({
    "monthly_usage_hours": xx1.ravel(),
    "late_payments": xx2.ravel(),
})

grid_prob = two_model.predict_proba(grid_df)[:, 1].reshape(xx1.shape)

plt.figure(figsize=(8, 5))
contour = plt.contour(
    xx1, xx2, grid_prob,
    levels=[0.5],
    linewidths=2
)

scatter = plt.scatter(
    X_train["monthly_usage_hours"],
    X_train["late_payments"],
    c=y_train,
    alpha=0.7
)

plt.xlabel("Monthly Usage Hours")
plt.ylabel("Late Payments")
plt.title("Logistic Regression Decision Boundary (p = 0.5)")
plt.show()

## Task 7.2 — Interpret the Boundary

Answer:

1. Which side of the line tends to contain more cancelled customers?
2. How does increased `late_payments` affect the decision?
3. How does increased `monthly_usage_hours` affect the decision?
4. Why is the boundary linear even though sigmoid itself is nonlinear?
5. What kind of class pattern would be difficult for this two-feature model to separate?

# Part VIII — Deliberate Debugging

A student writes:

```python
def bad_sigmoid(z):
    return 1 / (1 - np.exp(-z))
```

and later uses:

```python
w = w + alpha * dw
```

Two mathematical errors are present.

## Task 8.1 — Diagnose the Errors

Explain:

1. what is wrong with the sigmoid denominator;
2. what the correct formula is;
3. why ordinary gradient descent subtracts the gradient;
4. what may happen if the update repeatedly adds the gradient.

## Task 8.2 — Fix the Broken Functions

Complete:

In [ ]:
def fixed_sigmoid(z):
    # TODO: correct sigmoid.
    return None


def fixed_update(w, b, dw, db, alpha):
    # TODO: correct gradient-descent update.
    w_new = None
    b_new = None
    return w_new, b_new

In [ ]:
assert abs(fixed_sigmoid(0.0) - 0.5) < 1e-12

w_dbg, b_dbg = fixed_update(
    np.array([1.0, -1.0]),
    0.5,
    np.array([0.2, -0.3]),
    0.1,
    0.05
)

assert np.allclose(
    w_dbg,
    np.array([0.99, -0.985])
)
assert abs(b_dbg - 0.495) < 1e-12

print("Debugging tests passed.")

# Part IX — Personalized Probability Challenge

Your student-ID seed assigns one validation customer.

You must make a prediction **before** checking the model output.

In [ ]:
challenge_position = (SEED * 3) % len(X_valid)
challenge_index = X_valid.index[challenge_position]

challenge_row = X_valid.loc[[challenge_index]]

print("Assigned validation customer:")
display(challenge_row)

## Task 9.1 — Predict Before Running

Based only on the feature values and coefficient signs:

1. Do you expect cancellation probability to be above or below $0.5$?
2. Which two features are most influential in your reasoning?
3. Which feature pushes probability downward most strongly?
4. Which feature pushes probability upward most strongly?

Write your prediction before running the next cell.

In [ ]:
challenge_probability = logistic_model.predict_proba(
    challenge_row
)[0, 1]

challenge_prediction = int(
    challenge_probability >= 0.5
)

challenge_true = int(
    y_valid.loc[challenge_index]
)

print(f"Predicted cancellation probability: {challenge_probability:.4f}")
print("Predicted class:", challenge_prediction)
print("True class:", challenge_true)

## Task 9.2 — Analyze the Result

Answer:

1. Was your probability-direction prediction correct?
2. Was the class prediction correct?
3. Which coefficients explain the result most clearly?
4. If the prediction was wrong, does that automatically mean the model is poor?
5. Why must overall evaluation use many observations rather than one case?

# Part X — Final Test Check

The model structure is now fixed.

Use the test set only for a final check.

In [ ]:
test_prob = logistic_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

test_accuracy = accuracy_score(y_test, test_pred)
test_logloss = log_loss(y_test, test_prob)

print(f"Final test accuracy: {test_accuracy:.3f}")
print(f"Final test log loss: {test_logloss:.3f}")

## Task 10.1 — Final Generalization Statement

Write 3–5 sentences including:

- validation accuracy;
- validation log loss;
- test accuracy;
- test log loss;
- whether validation and test performance are reasonably consistent;
- one limitation of using threshold $0.5$ without considering application costs.

Do not change the model after inspecting the test result.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Why does logistic regression use sigmoid?
2. Explain the meaning of $z=w^\top x+b$.
3. Why does binary cross-entropy heavily penalize confident wrong predictions?
4. What does the quantity $p-y$ represent?
5. Explain one complete gradient-descent update.
6. Why does threshold $0.5$ correspond to $z=0$?
7. Why must the same training scaler be applied to validation/test data?
8. For your personalized customer, explain why the model predicted the probability it did.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What is the most important difference between linear regression output and logistic-regression output?
2. What did the manual update help you understand?
3. Why is a probability prediction more informative than a class label alone?
4. Why can coefficient signs be useful for interpretation?
5. What part of classification evaluation is still missing from this lab?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] your own student-ID-derived dataset;
- [ ] binary-classification formulation;
- [ ] completed sigmoid function;
- [ ] completed forward-pass function;
- [ ] manual forward-pass calculation;
- [ ] completed binary cross-entropy;
- [ ] completed one-example gradient function;
- [ ] manual full parameter update;
- [ ] fitted `scikit-learn` logistic-regression model;
- [ ] coefficient-sign interpretation;
- [ ] manual reproduction of one fitted probability;
- [ ] decision-boundary interpretation;
- [ ] corrected sigmoid and gradient-update debugging task;
- [ ] personalized probability challenge;
- [ ] final test check;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
x
\rightarrow
z
\rightarrow
p
\rightarrow
L
\rightarrow
p-y
\rightarrow
\nabla L
\rightarrow
\text{updated parameters}
}
$$

# Lab 6 Summary

You should now be able to explain and compute the internal learning process of logistic regression.

### Forward pass

$$
z=w^\top x+b
$$

$$
p=\sigma(z)
$$

### Binary cross-entropy

$$
L(y,p)
=
-\left[
y\log(p)+(1-y)\log(1-p)
\right]
$$

### Gradient signal

$$
\frac{\partial L}{\partial z}=p-y
$$

### Parameter update

$$
w\leftarrow w-\alpha(p-y)x
$$

$$
b\leftarrow b-\alpha(p-y)
$$

The next lab will shift the focus from **how the classifier learns** to **how its decisions should be evaluated**.

**Next lab:** Classification Evaluation, Thresholds, and Model Selection.